# 전처리 3 · 직접 풀어보는 연습문제 정답지

이 노트북은 `Preprocessing_3_Categorical_Encoding_Practice.ipynb`의 모든 연습문제에 대한 **표준 모범 코드와 결과 해석**을 제공합니다.


In [ ]:
# [실습 환경 준비]
# 이 셀은 인코딩 실습을 위한 범주형 고객 데이터를 생성합니다. 셀을 먼저 실행해주세요.
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder, LabelEncoder, OrdinalEncoder

cat_df = pd.DataFrame({
    "user_id": range(101, 109),
    "city": ["Seoul", "Busan", "Seoul", "Daegu", "Incheon", "Seoul", "Busan", "Gwangju"],
    "membership": ["Silver", "Gold", "Bronze", "VIP", "Silver", "VIP", "Bronze", "Gold"],
    "device": ["iOS", "Android", "Android", "iOS", "Windows", "iOS", "Android", "iOS"],
    "churn": ["Yes", "No", "No", "Yes", "No", "No", "Yes", "No"]
})


## 문제 1. pd.get_dummies를 활용한 원-핫 인코딩과 더미 함정(drop_first) 회피 [판다스 더미]

`cat_df[['device']]` 컬럼에 대해:
1. `pd.get_dummies()`를 적용하여 `dtype=int`로 변환된 더미 데이터프레임을 생성하세요.
2. 다중공선성(더미 함정, Dummy Variable Trap)을 회피하기 위해 `drop_first=True` 옵션을 적용한 결과를 비교 출력하세요.
3. 컬럼 수가 몇 개에서 몇 개로 줄었는지 확인하세요.


In [ ]:
# 문제 1 풀이
# 1. 기본 원-핫 인코딩
dummies_all = pd.get_dummies(cat_df["device"], prefix="dev", dtype=int)
print("=== 1. drop_first=False (전체 범주 더미) ===")
print(dummies_all.head(4))

# 2. drop_first=True 적용
dummies_drop = pd.get_dummies(cat_df["device"], prefix="dev", drop_first=True, dtype=int)
print("\n=== 2. drop_first=True (기준 범주 제외 더미) ===")
print(dummies_drop.head(4))

print(f"\n컬럼 수 변화: {dummies_all.shape[1]}개 -> {dummies_drop.shape[1]}개")


**결과 해석 / 확인한 점:**  
1. **더미 함정(Dummy Variable Trap)의 이해:** `Android`, `iOS`, `Windows` 3개 더미의 합은 항상 1이 되므로 완벽한 다중공선성이 발생합니다. 한 열을 제거(`drop_first=True`)해도 나머지 두 열이 0이면 제거된 범주임을 알 수 있으므로 정보 손실 없이 공선성을 방지합니다.
2. **dtype=int 명시:** 최신 Pandas에서는 불리언(`True/False`) 대신 `dtype=int`를 명시하여 정수 0과 1로 명확히 인코딩하는 것이 안전합니다.


## 문제 2. OneHotEncoder의 handle_unknown='ignore'를 통한 신규 범주 방어 [사이킷런 인코더]

훈련 데이터에는 없던 새로운 도시(예: `"Jeju"`, `"Ulsan"`)가 테스트 데이터에 등장할 때 오류 없이 방어해야 합니다.
1. `OneHotEncoder(handle_unknown='ignore', sparse_output=False)` 객체를 생성하고 `cat_df[['city']]`로 `.fit()`하세요.
2. 훈련된 인코더로 새로운 도시가 포함된 `new_cities = pd.DataFrame({'city': ['Seoul', 'Jeju', 'Busan', 'Ulsan']})`를 `.transform()`하세요.
3. 신규 도시들이 에러 없이 모두 0으로 인코딩되었는지 확인하세요.


In [ ]:
# 문제 2 풀이
# 1. 인코더 학습
ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
ohe.fit(cat_df[["city"]])

print("학습된 도시 범주 목록:", ohe.categories_[0])

# 2. 미학습 도시가 포함된 신규 데이터 변환
new_cities = pd.DataFrame({"city": ["Seoul", "Jeju", "Busan", "Ulsan"]})
transformed_new = ohe.transform(new_cities[["city"]])

df_new_ohe = pd.DataFrame(transformed_new, columns=ohe.get_feature_names_out(["city"]))
print("\n=== 미학습 도시 변환 결과 (Jeju, Ulsan 방어) ===")
print(pd.concat([new_cities, df_new_ohe], axis=1))


**결과 해석 / 확인한 점:**  
1. **실무 서빙 배포의 핵심:** `handle_unknown='ignore'`가 없을 경우 미학습 카테고리가 들어오면 `ValueError`로 전체 API 서버가 다운되지만, ignore 옵션 덕분에 모두 0인 더미 벡터로 안전하게 처리되었습니다.
2. **배포 안정성:** 서울과 부산은 정상 인코딩되고, 제주와 울산은 모든 더미 열이 0으로 처리되어 시스템 무중단 서빙이 가능합니다.


## 문제 3. LabelEncoder를 활용한 타겟 변수(churn)의 이진 수치화 [타겟 인코딩]

타겟 레이블인 `cat_df['churn']` ("Yes", "No")을 0과 1의 숫자로 변환합니다:
1. `LabelEncoder` 객체를 생성하고 `.fit_transform()`을 수행하여 `churn_encoded`를 생성하세요.
2. 어떤 문자열이 0이고 어떤 문자열이 1인지 `encoder.classes_` 속성을 통해 확인하세요.
3. 원래 문자열로 되돌리는 `.inverse_transform()` 결과를 검증하세요.


In [ ]:
# 문제 3 풀이
le = LabelEncoder()
cat_df["churn_encoded"] = le.fit_transform(cat_df["churn"])

print("=== 타겟 레이블 인코딩 결과 ===")
print(cat_df[["churn", "churn_encoded"]])

print(f"\n인코딩 클래스 매핑: {le.classes_}")
print(f"0 = '{le.classes_[0]}', 1 = '{le.classes_[1]}'")

# 역변환 검증
restored_churn = le.inverse_transform(cat_df["churn_encoded"])
print("역변환 일치 여부:", list(restored_churn) == list(cat_df["churn"]))


**결과 해석 / 확인한 점:**  
1. **사전순 정렬 원칙:** `LabelEncoder`는 알파벳 오름차순으로 번호를 매기므로 `No`가 0, `Yes`가 1로 자동 지정되었습니다.
2. **독립변수 사용 금지 경고:** `LabelEncoder`는 1차원 타겟 벡터 $y$에만 사용해야 하며, 독립변수 $X$에 적용하면 트리 기반 모형 외의 선형 모델에서 인위적 크기 서열이 왜곡을 유발하므로 주의해야 합니다.


## 문제 4. OrdinalEncoder를 통한 회원 등급(membership) 순서형 정수 인코딩 [순서형 인코딩]

회원 등급 `membership`은 명확한 서열 위계를 가집니다:
- 순서: `Bronze (0) < Silver (1) < Gold (2) < VIP (3)`
1. 위 서열 순서를 명시한 `categories` 리스트를 정의하고 `OrdinalEncoder`를 생성하세요.
2. `cat_df[['membership']]`을 변환하여 `membership_rank` 컬럼으로 추가하세요.
3. 의도한 서열(0~3)대로 정확히 매핑되었는지 확인하세요.


In [ ]:
# 문제 4 풀이
tier_hierarchy = [["Bronze", "Silver", "Gold", "VIP"]]
ordinal_enc = OrdinalEncoder(categories=tier_hierarchy)

cat_df["membership_rank"] = ordinal_enc.fit_transform(cat_df[["membership"]]).astype(int)

print("=== 순서형 인코딩 결과 ===")
print(cat_df[["user_id", "membership", "membership_rank"]])


**결과 해석 / 확인한 점:**  
1. **명시적 카테고리 계층 전달:** `categories=[['Bronze', 'Silver', 'Gold', 'VIP']]`로 명시함으로써 Bronze=0, Silver=1, Gold=2, VIP=3으로 의도된 순서가 완벽히 보존되었습니다.
2. **순서형 피처의 수학적 가치:** 순서형 인코딩을 적용하면 등급 간의 크기 관계를 단 1개의 컬럼으로 효율적으로 회귀 모델에 전달할 수 있습니다.


## 문제 5. 고카디널리티(High-Cardinality) 희소 범주 병합 [차원 축소]

범주형 컬럼에 고유값이 수십~수백 개일 때 원-핫 인코딩을 하면 컬럼 수가 폭증합니다.
1. `cat_df['city']`의 빈도수를 확인하세요.
2. 빈도수가 2회 미만인 희소 도시(`'Daegu'`, `'Incheon'`, `'Gwangju'`)를 `'Other'`로 통합하는 `city_grouped` 컬럼을 생성하세요.
3. 통합 후 고유 범주 수가 몇 개로 압축되었는지 확인하세요.


In [ ]:
# 문제 5 풀이
# 1. 도시별 빈도 확인
city_counts = cat_df["city"].value_counts()
print("도시별 빈도:\n", city_counts)

# 2. 2회 미만 희소 범주 식별
rare_cities = city_counts[city_counts < 2].index
print("\n희소 범주 목록 (2회 미만):", list(rare_cities))

# 3. 'Other'로 병합
cat_df["city_grouped"] = cat_df["city"].apply(lambda x: "Other" if x in rare_cities else x)

print("\n=== 희소 범주 통합 후 빈도분포 ===")
print(cat_df["city_grouped"].value_counts())
print(f"고유 범주 수 축소: {cat_df['city'].nunique()}개 -> {cat_df['city_grouped'].nunique()}개")


**결과 해석 / 확인한 점:**  
1. **차원의 저주(Curse of Dimensionality) 방어:** 표본 수가 적은 범주들을 'Other'로 묶어줌으로써 원-핫 인코딩 시 무의미하게 희소한 컬럼이 수십 개 생성되는 것을 방지합니다.
2. **과적합(Overfitting) 억제:** 단 1번만 등장하는 도시가 특정 패턴으로 오인되는 모델 과적합을 효과적으로 억제할 수 있습니다.


## 문제 6. 원-핫 인코딩 후 피처 매트릭스 결합 및 최종 정제 [데이터셋 준비]

머신러닝 입력용 최종 피처 매트릭스를 완성합니다:
1. `city_grouped`와 `device` 두 범주형 컬럼에 대해 `pd.get_dummies(..., drop_first=True)`를 적용하세요.
2. 수치형 피처인 `membership_rank`와 결합하여 최종 입력 데이터프레임 `X_encoded`를 생성하세요.
3. 모든 컬럼의 dtype이 수치형(`int` 또는 `float`)으로 통일되었는지 검증하세요.


In [ ]:
# 문제 6 풀이
# 1. 다중 범주 원-핫 인코딩
encoded_dummies = pd.get_dummies(cat_df[["city_grouped", "device"]], drop_first=True, dtype=int)

# 2. 수치형 피처 결합
X_encoded = pd.concat([cat_df[["membership_rank"]], encoded_dummies], axis=1)

print("=== 최종 인코딩 완료 피처 매트릭스 X ===")
print(X_encoded.head(5))
print("\n최종 컬럼별 dtypes:")
print(X_encoded.dtypes)
assert all(np.issubdtype(dt, np.number) for dt in X_encoded.dtypes), "비수치형 컬럼 잔여 오류!"
print("\n검증 성공: 모든 피처가 100% 수치형으로 완벽 변환되었습니다.")


**결과 해석 / 확인한 점:**  
1. **머신러닝 투입 준비 완료:** 범주형 문자열이 완전히 배제되고 모든 피처가 수치형 더미 및 랭크 변수로 정제되어 사이킷런 모든 모델에 즉시 학습 가능한 상태가 되었습니다.
2. **다중공선성 방어:** `drop_first=True`가 적용되어 완전한 선형 독립성을 만족합니다.
